In [2]:
import os
import pandas as pd
import chromadb

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Define database path
CHROMA_PATH = "../chroma_db"

# Load embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Connect to existing ChromaDB
vector_store = Chroma(
    collection_name="ai_documents",
    embedding_function=embedding_model,
    persist_directory=CHROMA_PATH
)

print("Connected to ChromaDB successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connected to ChromaDB successfully!


In [4]:
# Check stored documents
collection = vector_store._collection

print("Collection Name:", collection.name)
print("Total Documents:", collection.count())

Collection Name: ai_documents
Total Documents: 20


In [4]:
from pathlib import Path
from langchain_core.documents import Document

# Read documents from the documents folder
documents_path = Path("../dataset/documents")

documents = []

for i, file in enumerate(sorted(documents_path.glob("*.txt"))):
    content = file.read_text(encoding="utf-8")

    documents.append(
        Document(
            page_content=content,
            metadata={"source": file.name}
        )
    )

# Add documents to ChromaDB
ids = [f"week12_doc_{i}" for i in range(len(documents))]

vector_store.add_documents(
    documents=documents,
    ids=ids
)

print("New documents added successfully!")
print("New documents:", len(documents))
print("Total documents:", vector_store._collection.count())

New documents added successfully!
New documents: 10
Total documents: 20


In [5]:
queries = [
    "What is artificial intelligence?",
    "How do computers learn from data?",
    "Explain deep learning and neural networks.",
    "How can computers understand human language?",
    "How do machines recognize images?",
    "What is the role of robots in automation?",
    "How can we protect computer systems from attacks?",
    "What are the benefits of cloud computing?",
    "How is data science used to analyze information?",
    "How do chatbots communicate with people?",
    "What is the difference between AI and machine learning?",
    "How are neural networks used in technology?",
    "How does natural language processing work?",
    "Why is cybersecurity important?",
    "How does technology help in decision making?"
]

print("Total Queries:", len(queries))

for i, query in enumerate(queries, 1):
    print(f"{i}. {query}")

Total Queries: 15
1. What is artificial intelligence?
2. How do computers learn from data?
3. Explain deep learning and neural networks.
4. How can computers understand human language?
5. How do machines recognize images?
6. What is the role of robots in automation?
7. How can we protect computer systems from attacks?
8. What are the benefits of cloud computing?
9. How is data science used to analyze information?
10. How do chatbots communicate with people?
11. What is the difference between AI and machine learning?
12. How are neural networks used in technology?
13. How does natural language processing work?
14. Why is cybersecurity important?
15. How does technology help in decision making?


In [6]:
search_results = []

for query in queries:
    results = vector_store.similarity_search_with_score(
        query,
        k=5
    )

    for rank, (doc, score) in enumerate(results, 1):
        search_results.append({
            "Query": query,
            "Rank": rank,
            "Document": doc.page_content,
            "Source": doc.metadata.get("source", "Unknown"),
            "Similarity_Score": round(float(score), 4)
        })

print("Semantic search completed!")
print("Total retrieved results:", len(search_results))

Semantic search completed!
Total retrieved results: 75


In [7]:
for query in queries:
    print("\nQuery:", query)
    print("-" * 60)

    results = vector_store.similarity_search_with_score(
        query,
        k=5
    )

    for rank, (doc, score) in enumerate(results, 1):
        print(f"\nRank: {rank}")
        print("Document:", doc.page_content)
        print("Distance Score:", round(float(score), 4))


Query: What is artificial intelligence?
------------------------------------------------------------

Rank: 1
Document: Artificial Intelligence (AI)

Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence. AI systems can learn from data, solve problems, understand languages, and make decisions. Examples of AI applications include virtual assistants, chatbots, recommendation systems, and self-driving cars. AI is widely used in education, healthcare, banking, and transportation.
Distance Score: 0.363

Rank: 2
Document: Artificial Intelligence enables machines to perform tasks that normally require human intelligence.
Distance Score: 0.5544

Rank: 3
Document: Robotics and Artificial Intelligence

Robotics combines engineering and Artificial Intelligence to create intelligent machines. AI-powered robots can perform tasks automatically, recognize objects, and make decisions. Robots are used in industries, hosp

In [8]:
# Convert search results into DataFrame
results_df = pd.DataFrame(search_results)

# Save results
results_df.to_csv(
    "../dataset/search_results.csv",
    index=False
)

print("Search results saved successfully!")
print(results_df.head())

Search results saved successfully!
                              Query  Rank  \
0  What is artificial intelligence?     1   
1  What is artificial intelligence?     2   
2  What is artificial intelligence?     3   
3  What is artificial intelligence?     4   
4  What is artificial intelligence?     5   

                                            Document          Source  \
0  Artificial Intelligence (AI)\n\nArtificial Int...   document1.txt   
1  Artificial Intelligence enables machines to pe...   document1.txt   
2  Robotics and Artificial Intelligence\n\nRoboti...   document7.txt   
3  Robotics combines artificial intelligence, sen...   document6.txt   
4  Applications of Artificial Intelligence\n\nArt...  document10.txt   

   Similarity_Score  
0            0.3630  
1            0.5544  
2            0.7341  
3            0.8521  
4            0.8639  


In [9]:
# Create similarity score DataFrame
scores_df = results_df[
    ["Query", "Rank", "Source", "Similarity_Score"]
]

# Save similarity scores
scores_df.to_csv(
    "../dataset/similarity_scores.csv",
    index=False
)

print("Similarity scores saved successfully!")
print(scores_df.head())

Similarity scores saved successfully!
                              Query  Rank          Source  Similarity_Score
0  What is artificial intelligence?     1   document1.txt            0.3630
1  What is artificial intelligence?     2   document1.txt            0.5544
2  What is artificial intelligence?     3   document7.txt            0.7341
3  What is artificial intelligence?     4   document6.txt            0.8521
4  What is artificial intelligence?     5  document10.txt            0.8639


In [10]:
# Create query DataFrame
queries_df = pd.DataFrame({
    "Query_ID": range(1, len(queries) + 1),
    "User_Query": queries
})

# Save queries
queries_df.to_csv(
    "../dataset/queries.csv",
    index=False
)

print("Queries saved successfully!")
print(queries_df)

Queries saved successfully!
    Query_ID                                         User_Query
0          1                   What is artificial intelligence?
1          2                  How do computers learn from data?
2          3         Explain deep learning and neural networks.
3          4       How can computers understand human language?
4          5                  How do machines recognize images?
5          6          What is the role of robots in automation?
6          7  How can we protect computer systems from attacks?
7          8          What are the benefits of cloud computing?
8          9   How is data science used to analyze information?
9         10           How do chatbots communicate with people?
10        11  What is the difference between AI and machine ...
11        12        How are neural networks used in technology?
12        13         How does natural language processing work?
13        14                    Why is cybersecurity important?
14        15

In [11]:
# Keyword-based search function
def keyword_search(query, documents, top_k=5):
    query_words = set(query.lower().split())
    results = []

    for doc in documents:
        content = doc.page_content.lower()
        score = sum(word in content for word in query_words)

        if score > 0:
            results.append((doc, score))

    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_k]


# Compare both search methods
comparison_query = "How do computers learn from data?"

print("Query:", comparison_query)

print("\nKeyword Search Results:")
for doc, score in keyword_search(comparison_query, documents):
    print("Document:", doc.page_content)
    print("Keyword Score:", score)

print("\nSemantic Search Results:")
for doc, score in vector_store.similarity_search_with_score(
    comparison_query, k=5
):
    print("Document:", doc.page_content)
    print("Distance Score:", round(float(score), 4))

Query: How do computers learn from data?

Keyword Search Results:
Document: Machine Learning allows computers to learn patterns from data and make predictions.
Keyword Score: 3
Document: Cybersecurity protects computers, networks, and data from unauthorized access and attacks.
Keyword Score: 2
Document: Deep Learning uses neural networks with multiple layers to process complex information.
Keyword Score: 1
Document: Natural Language Processing helps computers understand and generate human language.
Keyword Score: 1

Semantic Search Results:
Document: Machine Learning allows computers to learn patterns from data and make predictions.
Distance Score: 0.6734
Document: Data Science uses statistics, programming, and data analysis to discover useful information.
Distance Score: 1.0207
Document: Computer Vision enables machines to understand images and videos.
Distance Score: 1.0716
Document: Machine Learning

Machine Learning is a subset of Artificial Intelligence that allows computers to le

In [ ]:
# Accept query from user
user_query = input("Enter your search query: ")

# Perform semantic search
results = vector_store.similarity_search_with_score(
    user_query,
    k=5
)

print("\nTop 5 Relevant Documents:")

for rank, (doc, score) in enumerate(results, 1):
    print(f"\nRank: {rank}")
    print("Document:", doc.page_content)
    print("Distance Score:", round(float(score), 4))

In [5]:
queries = [
    "What is artificial intelligence?",
    "How do computers learn from data?",
    "Explain deep learning and neural networks.",
    "How can computers understand human language?",
    "How do machines recognize images?",
    "What is the role of robots in automation?",
    "How can we protect computer systems from attacks?",
    "What are the benefits of cloud computing?",
    "How is data science used to analyze information?",
    "How do chatbots communicate with people?",
    "What is the difference between AI and machine learning?",
    "How are neural networks used in technology?",
    "How does natural language processing work?",
    "Why is cybersecurity important?",
    "How does technology help in decision making?"
]

print("Total Queries:", len(queries))

Total Queries: 15


In [6]:
search_results = []

for query in queries:
    results = vector_store.similarity_search_with_score(
        query,
        k=5
    )

    for rank, (doc, score) in enumerate(results, 1):
        search_results.append({
            "Query": query,
            "Rank": rank,
            "Document": doc.page_content,
            "Source": doc.metadata.get("source", "Unknown"),
            "Similarity_Score": round(float(score), 4)
        })

print("Semantic search completed!")
print("Total retrieved results:", len(search_results))

Semantic search completed!
Total retrieved results: 75


In [7]:
for query in queries:
    print("\nQuery:", query)
    print("-" * 50)

    results = vector_store.similarity_search_with_score(
        query, k=5
    )

    for rank, (doc, score) in enumerate(results, 1):
        print(f"\nRank: {rank}")
        print("Document:", doc.page_content)
        print("Distance Score:", round(float(score), 4))


Query: What is artificial intelligence?
--------------------------------------------------

Rank: 1
Document: Artificial Intelligence (AI)

Artificial Intelligence is a branch of computer science that enables machines to perform tasks that normally require human intelligence. AI systems can learn from data, solve problems, understand languages, and make decisions. Examples of AI applications include virtual assistants, chatbots, recommendation systems, and self-driving cars. AI is widely used in education, healthcare, banking, and transportation.
Distance Score: 0.363

Rank: 2
Document: Artificial Intelligence enables machines to perform tasks that normally require human intelligence.
Distance Score: 0.5544

Rank: 3
Document: Robotics and Artificial Intelligence

Robotics combines engineering and Artificial Intelligence to create intelligent machines. AI-powered robots can perform tasks automatically, recognize objects, and make decisions. Robots are used in industries, hospitals, man

In [8]:
results_df = pd.DataFrame(search_results)

results_df.to_csv(
    "../dataset/search_results.csv",
    index=False
)

print("Search results saved successfully!")
print("Total Results:", len(results_df))

Search results saved successfully!
Total Results: 75


In [9]:
scores_df = results_df[
    ["Query", "Rank", "Source", "Similarity_Score"]
]

scores_df.to_csv(
    "../dataset/similarity_scores.csv",
    index=False
)

print("Similarity scores saved successfully!")
print("Total Scores:", len(scores_df))

Similarity scores saved successfully!
Total Scores: 75


In [10]:
queries_df = pd.DataFrame({
    "Query_ID": range(1, len(queries) + 1),
    "User_Query": queries
})

queries_df.to_csv(
    "../dataset/queries.csv",
    index=False
)

print("Queries saved successfully!")
print("Total Queries:", len(queries_df))

Queries saved successfully!
Total Queries: 15


In [11]:
from langchain_core.documents import Document

# Load all stored documents
stored_data = vector_store._collection.get(
    include=["documents", "metadatas"]
)

all_documents = [
    Document(page_content=content, metadata=metadata or {})
    for content, metadata in zip(
        stored_data["documents"],
        stored_data["metadatas"]
    )
]

# Keyword search
def keyword_search(query, documents, top_k=5):
    query_words = set(query.lower().split())
    results = []

    for doc in documents:
        content = doc.page_content.lower()
        score = sum(word in content for word in query_words)

        if score > 0:
            results.append((doc, score))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]


# Compare both search methods
comparison_query = "How do computers learn from data?"

print("Query:", comparison_query)

print("\nKeyword Search Results:")
for doc, score in keyword_search(
    comparison_query, all_documents
):
    print(doc.page_content, "| Score:", score)

print("\nSemantic Search Results:")
for doc, score in vector_store.similarity_search_with_score(
    comparison_query, k=5
):
    print(doc.page_content, "| Distance:", round(float(score), 4))

Query: How do computers learn from data?

Keyword Search Results:
Machine Learning

Machine Learning is a subset of Artificial Intelligence that allows computers to learn from data without being explicitly programmed. It uses algorithms to identify patterns and make predictions. Machine Learning is used in email spam detection, weather forecasting, fraud detection, and product recommendations. The three main types are supervised learning, unsupervised learning, and reinforcement learning. | Score: 3
Neural Networks

Neural Networks are computing systems inspired by the human brain. They consist of interconnected nodes called neurons. These networks learn patterns from data and help computers perform tasks such as image classification, speech recognition, and prediction. Neural networks are an important part of Deep Learning. | Score: 3
Machine Learning allows computers to learn patterns from data and make predictions. | Score: 3
Artificial Intelligence (AI)

Artificial Intelligence is 

In [ ]:

import pandas as pd
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Database path
CHROMA_PATH = "chroma_db"

# Load embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Connect to ChromaDB
vector_store = Chroma(
    collection_name="ai_documents",
    embedding_function=embedding_model,
    persist_directory=CHROMA_PATH
)

print("Semantic Document Search System")

# Accept user query
query = input("Enter your search query: ")

# Retrieve Top-5 documents
results = vector_store.similarity_search_with_score(
    query, k=5
)

print("\nTop 5 Relevant Documents:")

for rank, (doc, score) in enumerate(results, 1):
    print(f"\nRank: {rank}")
    print("Document:", doc.page_content)
    print("Distance Score:", round(float(score), 4))
